# Feature-conditioned perturbation encoder — mini demo (LARRY)

pertTF's default perturbation encoder, **`PertLabelEncoder`**, is an `nn.Embedding`
indexed by genotype. A perturbation that is **absent from training** therefore only
ever has a *random, untrained* embedding row — so the model structurally cannot make
an informed prediction for an unseen condition (zero-shot / leave-one-condition-out).

**`FeaturePertEncoder`** (PR #40) replaces that lookup with a small trainable MLP on a
*fixed per-perturbation feature matrix*. Because the embedding is a smooth function of
perturbation features, shared across all conditions through one MLP, a condition never
seen in training still gets a meaningful, training-informed embedding.

This notebook:
1. loads a tiny **LARRY**-derived AnnData,
2. holds out one cytokine condition (`SCF`) entirely from training,
3. runs a real **mini train/validation loop** with both encoders, and
4. shows the **zero-shot** property directly: the held-out condition is learnable under
   `FeaturePertEncoder` but provably frozen under `PertLabelEncoder`.

> All heavy lifting reuses helper functions from `feature_pert_encoder_demo.py` (same
> directory) so the notebook stays readable; open that file to see the details.

In [1]:
import os, sys, warnings
warnings.filterwarnings("ignore")
sys.path.insert(0, os.path.abspath("."))                 # find feature_pert_encoder_demo.py
PERTTF_REPO = os.environ.get("PERTTF_REPO", "/content/pertTF")
if PERTTF_REPO not in sys.path:
    sys.path.insert(0, PERTTF_REPO)

import numpy as np, scanpy as sc, torch
import feature_pert_encoder_demo as demo   # build_pert_features, make_config, build_model, train_model, ...
from perttf.model.train_data_gen import produce_training_datasets

SEED, HELD_OUT = 0, "SCF"
torch.manual_seed(SEED); np.random.seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

⚠️ Flash Attention not installed. Model will use standard attention.


device: cuda


## 1. The mini LARRY dataset

LARRY (Weinreb et al. 2020) is an HSPC cytokine lineage-tracing dataset. We map its
columns to the pertTF convention:

| LARRY column | pertTF role |
|---|---|
| `cell_type` | `obs['celltype']` |
| `cytokine_condition` (baseline → `WT`) | `obs['genotype']` (the "perturbation") |

We **download the prepared mini dataset directly from Google Drive** (1,800 cells × 400
HVGs across 6 conditions, ~6 MB). It was originally produced by `make_larry_mini.py`
from the 1.15 GB read-only source `larry_cytokine.h5ad`, so downloading the shared copy
means this notebook needs no access to `/content/barcodes`.

In [2]:
H5AD = "larry_mini.h5ad"
GDRIVE_ID = "1NIzP3rBdFWiqrjpVPydhw1h9fYP5ZeSs"   # shared larry_mini.h5ad on Google Drive
if not os.path.exists(H5AD):
    import gdown
    gdown.download(f"https://drive.google.com/uc?id={GDRIVE_ID}", H5AD, quiet=False)

adata = sc.read_h5ad(H5AD)
adata.obs["genotype"] = adata.obs["genotype"].astype(str)
adata.obs["celltype"] = adata.obs["celltype"].astype(str)
if "X_binned" not in adata.layers:
    adata.layers["X_binned"] = adata.X.copy()
print(adata)
print("\ngenotype counts:\n", adata.obs["genotype"].value_counts())

Downloading...
From: https://drive.google.com/uc?id=1NIzP3rBdFWiqrjpVPydhw1h9fYP5ZeSs
To: /content/perttf_dev/larry_mini.h5ad


  0%|          | 0.00/5.93M [00:00<?, ?B/s]

100%|██████████| 5.93M/5.93M [00:00<00:00, 372MB/s]

AnnData object with n_obs × n_vars = 1800 × 400
    obs: 'genotype', 'celltype', 'time_point', 'is_baseline'
    layers: 'X_binned'

genotype counts:
 genotype
full cocktail    300
Epo              300
G-csf            300
SCF              300
M-csf            300
WT               300
Name: count, dtype: int64


## 2. Label maps and the perturbation feature matrix

We build a **union** `genotype_to_index` over *all* conditions — including the held-out
one — so `SCF` gets an index (and a feature row) even though it contributes no training
cells. `pert_features` has shape `(n_genotypes, feat_dim)`, where **row `i` is the
feature vector for the genotype whose index is `i`**.

Here we proxy "perturbation features" with each condition's *pseudobulk molecular
signature* (mean log-expression → PCA). In a real zero-shot setting these features come
from prior knowledge that also exists for unseen perturbations (gene/drug identity
embeddings, target/receptor genes, chemical structure). The only requirement is that a
feature row can be defined for a novel condition without having trained on it.

In [3]:
genotype_to_index = {g: i for i, g in enumerate(sorted(adata.obs["genotype"].unique()))}
cell_type_to_index = {c: i for i, c in enumerate(sorted(adata.obs["celltype"].unique()))}
held_idx = genotype_to_index[HELD_OUT]
print("genotype_to_index:", genotype_to_index)

pert_features = demo.build_pert_features(adata, genotype_to_index, feat_dim=32)
print("pert_features:", pert_features.shape, "| held-out row index:", held_idx)

genotype_to_index: {'Epo': 0, 'G-csf': 1, 'M-csf': 2, 'SCF': 3, 'WT': 4, 'full cocktail': 5}
pert_features: (6, 32) | held-out row index: 3


## 3. Hold out a condition and build the training datasets

We train on every condition **except** `SCF`. The genotype index space still includes
`SCF`, so the model allocates an encoder slot for it — but it never sees an `SCF` cell.

`make_config()` sets `perturbation_input=True`: the perturbation embedding is injected
into the transformer, so the masked-expression reconstruction loss flows back through
the perturbation encoder (otherwise the encoder would get no gradient at all).

In [4]:
train_adata = adata[adata.obs["genotype"] != HELD_OUT].copy()
print(f"training cells: {train_adata.n_obs} (held out all {adata.n_obs - train_adata.n_obs} '{HELD_OUT}' cells)")

config = demo.make_config()
data_gen = produce_training_datasets(
    train_adata, config,
    next_cell_pred=config.next_cell_pred_type,
    genotype_to_index=genotype_to_index,
    cell_type_to_index=cell_type_to_index,
)
print("n_perturb =", data_gen["n_perturb"], "(index space includes the unseen condition)")

wandb: WARNING `start_method` is deprecated and will be removed in a future version of wandb. This setting is currently non-functional and safely ignored.


training cells: 1500 (held out all 300 'SCF' cells)
{'seed': 0, 'dataset_name': 'larry_mini_feature_pert_demo', 'do_train': True, 'GEPC': False, 'ecs_thres': 0.0, 'ecs_weight': 1.0, 'dab_weight': 0.0, 'this_weight': 1.0, 'next_weight': 0, 'next_cell_pred_type': 'identity', 'cell_type_classifier': True, 'cell_type_classifier_weight': 1.0, 'genotype_classifier': False, 'perturbation_classifier_weight': 0.0, 'ps_weight': 0.0, 'perturbation_input': True, 'CCE': False, 'mask_ratio': 0.15, 'epochs': 4, 'n_bins': 0, 'lr': 0.001, 'batch_size': 64, 'layer_size': 32, 'nlayers': 2, 'nhead': 2, 'dropout': 0.2, 'schedule_ratio': 0.97, 'schedule_interval': 1, 'log_interval': 50, 'fast_transformer': False, 'pre_norm': False, 'amp': False, 'do_sample_in_train': False, 'ADV': False, 'adv_weight': 0, 'adv_E_delay_epochs': 0, 'adv_D_delay_epochs': 0, 'lr_ADV': 0.001, 'DSBN': False, 'per_seq_batch_sample': False, 'use_batch_label': False, 'explicit_zero_prob': False, 'n_hvg': 400, 'mask_value': -1, 'pad_v

## 4. Train with `FeaturePertEncoder`

Passing `pert_features=...` to `PerturbationTFModel`/`HFPerturbationTFModel` is the only
switch needed — it selects `FeaturePertEncoder` instead of `PertLabelEncoder`. We record
the encoder's output for the held-out condition *before* and *after* training.

In [5]:
torch.manual_seed(SEED)
model_feat = demo.build_model(data_gen, config, pert_features=pert_features)
print("encoder:", type(model_feat.pert_encoder).__name__)

emb_feat_before = demo.pert_embeddings(model_feat, torch.device("cpu"))
model_feat = demo.train_model(model_feat, data_gen, config, device, "FeaturePertEncoder")
emb_feat_after = demo.pert_embeddings(model_feat, device)

encoder: FeaturePertEncoder



=== training [FeaturePertEncoder] (FeaturePertEncoder) ===


  epoch 1/4 | val mse 0.1442 | val celltype-CE 1.5987 | val genotype-CE 0.0000


  epoch 2/4 | val mse 0.1273 | val celltype-CE 1.4752 | val genotype-CE 0.0000


  epoch 3/4 | val mse 0.1426 | val celltype-CE 1.4074 | val genotype-CE 0.0000


  epoch 4/4 | val mse 0.1287 | val celltype-CE 1.3247 | val genotype-CE 0.0000


## 5. Baseline: the default `PertLabelEncoder`

Same data, same loop, but `pert_features=None`. Its only learnable state for a condition
is one `nn.Embedding` row; we snapshot the held-out row before/after training.

In [6]:
torch.manual_seed(SEED)
model_lbl = demo.build_model(data_gen, config, pert_features=None)
print("encoder:", type(model_lbl.pert_encoder).__name__)

raw_lbl_before = model_lbl.pert_encoder.embedding.weight[held_idx].detach().cpu().numpy().copy()
model_lbl = demo.train_model(model_lbl, data_gen, config, device, "PertLabelEncoder")
raw_lbl_after = model_lbl.pert_encoder.embedding.weight[held_idx].detach().cpu().numpy().copy()

encoder: PertLabelEncoder

=== training [PertLabelEncoder] (PertLabelEncoder) ===


  epoch 1/4 | val mse 0.1413 | val celltype-CE 1.6201 | val genotype-CE 0.0000


  epoch 2/4 | val mse 0.1406 | val celltype-CE 1.4908 | val genotype-CE 0.0000


  epoch 3/4 | val mse 0.1408 | val celltype-CE 1.4404 | val genotype-CE 0.0000


  epoch 4/4 | val mse 0.1296 | val celltype-CE 1.4147 | val genotype-CE 0.0000


## 6. Zero-shot check

Did training teach the model anything about a condition with **zero training cells**?

In [7]:
l2 = lambda a, b: float(np.linalg.norm(a - b))

print(f"Unseen condition '{HELD_OUT}' (index {held_idx})\n")
print("PertLabelEncoder  — only state = one nn.Embedding row")
print(f"   ||row_after - row_before||  = {l2(raw_lbl_after, raw_lbl_before):.6f}   (no gradient -> frozen at init)\n")
print("FeaturePertEncoder — embedding = sharedMLP(features), no per-condition row")
print(f"   ||emb_after - emb_before||  = {l2(emb_feat_after[held_idx], emb_feat_before[held_idx]):.6f}   (informed by training on other conditions)")

Unseen condition 'SCF' (index 3)

PertLabelEncoder  — only state = one nn.Embedding row
   ||row_after - row_before||  = 0.000000   (no gradient -> frozen at init)

FeaturePertEncoder — embedding = sharedMLP(features), no per-condition row
   ||emb_after - emb_before||  = 0.910903   (informed by training on other conditions)


**Takeaway.** The `PertLabelEncoder` row for the unseen condition is *exactly* unchanged
(`0.000000`) — it receives no gradient, so the model is no better than random init for
that condition. The `FeaturePertEncoder` embedding for the same unseen condition moves
substantially, because the shared MLP is trained on the other conditions and the unseen
condition's embedding is a smooth function of its features.

### Using it in your own code

```python
from perttf.model.pertTF import PerturbationTFModel        # or HFPerturbationTFModel
# pert_features: (n_pert, feat_dim), row i = features for genotype index i.
# Assign held-out / novel conditions an index + feature row in genotype_to_index.
model = PerturbationTFModel(..., pert_features=pert_features)
```

For the HuggingFace path, `pert_features` is forwarded by `HFPerturbationTFModel.__init__`,
saved by `save_pretrained` (in `running_parameters.pt`), and restored by `from_pretrained`,
so the feature encoder survives a full save/load round-trip.